In [1]:
%%capture
!pip install --upgrade unsloth
!pip install transformers peft datasets trl -q

In [2]:
import os, sys, json, time, re, warnings, logging, torch
from pathlib import Path

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')
print(f'PyTorch: {torch.__version__}')

GPU: NVIDIA A100-SXM4-40GB
VRAM: 39.5 GB
PyTorch: 2.10.0+cu128


In [ ]:
# ====== SHARED CONFIG — chỉnh ở đây ======
CONFIG = {
    # Kaggle API
    'kaggle_token': 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx',
    'kaggle_username': 'thnhcngl',

    # Kaggle datasets
    'ds_sft_ckpt': 'thnhcngl/dvsktt-sft-best-checkpoint',
    'ds_sft_data': 'thnhcngl/dvsktt-ner-sft',
    'ds_han_data': 'thnhcngl/dvsktt-han-pretrain',

    # Local paths
    'data_dir':    '/content/data',
    'work_dir':    '/content/drive/MyDrive/dvsktt_ner',
    'ckpt_dir':    '/content/drive/MyDrive/dvsktt_ner/checkpoints',
    'result_dir':  '/content/drive/MyDrive/dvsktt_ner/results',
    'log_dir':     '/content/drive/MyDrive/dvsktt_ner/logs',

    # Model
    'base_model':   'unsloth/qwen2.5-7b-unsloth-bnb-4bit',
    'max_seq_len':  512,
    'lora_rank':    16,
    'lora_alpha':   32,
    'lora_dropout': 0.05,

    # Pretrain
    'pretrain_lr':     5e-5,
    'pretrain_epochs': 3,
    'pretrain_batch':  2,
    'pretrain_sample': 5000,
    'pretrain_grad_accum': 4,

    # SFT
    'sft_lr':          5e-5,
    'sft_epochs':      3,
    'sft_batch':       1,
    'sft_grad_accum':  4,

    # Evaluate
    'eval_batch':      4,
    'max_new_tokens':  600,
    'save_every':      50,

    # Resume
    'resume_from_step': 0,  # đặt số step để resume, 0 = train từ đầu
}

# Tạo thư mục
for d in ['data_dir','work_dir','ckpt_dir','result_dir','log_dir']:
    os.makedirs(CONFIG[d], exist_ok=True)

print('Config loaded. Dirs created.')
print(f"Work dir: {CONFIG['work_dir']}")

In [8]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
    print('Drive mounted.')
else:
    print('Drive already mounted.')

Drive already mounted.


In [9]:
# Setup Kaggle API
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({
        'username': CONFIG['kaggle_username'],
        'key': CONFIG['kaggle_token']
    }, f)
os.system('chmod 600 /root/.kaggle/kaggle.json')
print('Kaggle API configured.')
!kaggle --version

Kaggle API configured.
Kaggle CLI 2.0.2


In [11]:
import os, glob

# Tìm tất cả files vừa download
print("=== /content/data ===")
for item in os.listdir('/content/data'):
    print(f"  {item}")
    sub = f'/content/data/{item}'
    if os.path.isdir(sub):
        for f in os.listdir(sub):
            print(f"    {f}")

=== /content/data ===


In [12]:
import subprocess

def download_dataset(ds_name, data_dir):
    name = ds_name.split('/')[-1]
    dest = f'{data_dir}/{name}'

    if os.path.exists(dest) and len(os.listdir(dest)) > 0:
        print(f'Already exists: {dest}')
        return dest

    os.makedirs(dest, exist_ok=True)
    print(f'Downloading {ds_name}...')

    # Download với subprocess để thấy lỗi
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', ds_name, '-p', data_dir],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print(f'ERROR: {result.stderr}')
        return None

    # Tìm file zip
    zip_file = f'{data_dir}/{name}.zip'
    if not os.path.exists(zip_file):
        # Tìm zip file khác
        zips = [f for f in os.listdir(data_dir) if f.endswith('.zip')]
        print(f'Found zips: {zips}')
        if zips:
            zip_file = f'{data_dir}/{zips[0]}'

    # Unzip
    result2 = subprocess.run(
        ['unzip', '-q', zip_file, '-d', dest],
        capture_output=True, text=True
    )
    if result2.returncode != 0:
        print(f'Unzip error: {result2.stderr}')

    os.system(f'rm -f {zip_file}')

    print(f'Done: {dest}')
    for f in os.listdir(dest):
        print(f'  {f}')
    return dest

sft_ckpt_path = download_dataset(CONFIG['ds_sft_ckpt'], CONFIG['data_dir'])
sft_data_path = download_dataset(CONFIG['ds_sft_data'], CONFIG['data_dir'])
han_data_path = download_dataset(CONFIG['ds_han_data'], CONFIG['data_dir'])

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata

ERROR: 
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata

ERROR: 
403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/GetDatasetMetadata

ERROR: 


In [13]:
# ====== Logger — ghi log ra file + console ======
class Logger:
    def __init__(self, log_path):
        self.log_path = log_path
        self.start    = time.time()
        os.makedirs(os.path.dirname(log_path), exist_ok=True)
        with open(log_path, 'a') as f:
            f.write(f'\n===== Session started: {time.strftime("%Y-%m-%d %H:%M:%S")} =====\n')

    def log(self, msg, also_print=True):
        elapsed = time.time() - self.start
        line    = f'[{elapsed:>8.1f}s] {msg}'
        with open(self.log_path, 'a') as f:
            f.write(line + '\n')
        if also_print:
            print(line)

    def save_state(self, state, name):
        path = os.path.join(os.path.dirname(self.log_path), f'{name}.json')
        with open(path, 'w') as f:
            json.dump(state, f, ensure_ascii=False, indent=2)
        self.log(f'State saved: {path}')
        return path

    def load_state(self, name):
        path = os.path.join(os.path.dirname(self.log_path), f'{name}.json')
        if os.path.exists(path):
            with open(path) as f:
                state = json.load(f)
            self.log(f'State loaded: {path}')
            return state
        return None

print('Logger ready.')

Logger ready.


## Load Model + LoRA

In [14]:
import random
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup
import torch.optim as optim
from unsloth import FastLanguageModel

logger = Logger(f"{CONFIG['log_dir']}/pretrain.log")

# Check resume
resume_state = logger.load_state('pretrain_state')
RESUME_STEP  = resume_state['step'] if resume_state else CONFIG['resume_from_step']
if RESUME_STEP > 0:
    logger.log(f'Resuming from step {RESUME_STEP}')
else:
    logger.log('Starting fresh pretrain')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[     0.0s] Starting fresh pretrain


In [15]:
# Load base model hoặc resume checkpoint
pretrain_ckpt = f"{CONFIG['ckpt_dir']}/pretrain_final"

if RESUME_STEP > 0 and os.path.exists(pretrain_ckpt):
    logger.log(f'Loading pretrain checkpoint: {pretrain_ckpt}')
    model_path = pretrain_ckpt
else:
    logger.log(f"Loading base model: {CONFIG['base_model']}")
    model_path = CONFIG['base_model']

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_path,
    max_seq_length = CONFIG['max_seq_len'],
    load_in_4bit   = True,
    dtype          = None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = CONFIG['lora_rank'],
    lora_alpha     = CONFIG['lora_alpha'],
    lora_dropout   = CONFIG['lora_dropout'],
    target_modules = ['q_proj','k_proj','v_proj','o_proj',
                      'gate_proj','up_proj','down_proj'],
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = 42,
)
logger.log('Model loaded!')
model.print_trainable_parameters()

[     0.0s] Loading base model: unsloth/qwen2.5-7b-unsloth-bnb-4bit
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json:   0%|          | 0.00/106k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/172 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


[    59.9s] Model loaded!
trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [ ]:
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)

# Ghi trực tiếp kaggle.json
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({
        "username": "thnhcngl",
        "key": "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
    }, f)

os.system('chmod 600 /root/.kaggle/kaggle.json')
print('Kaggle configured!')

# Test download
import subprocess
result = subprocess.run(
    ['kaggle', 'datasets', 'download', 'thnhcngl/dvsktt-han-pretrain',
     '-p', '/content/data', '--unzip'],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [22]:
import os, subprocess

# Check xem data có chưa
print("=== /content/data ===")
if os.path.exists('/content/data'):
    for item in os.listdir('/content/data'):
        print(f"  {item}")
else:
    print("Empty!")

# Download lại tất cả
datasets = [
    ('thnhcngl/dvsktt-han-pretrain',        'dvsktt-han-pretrain'),
    ('thnhcngl/dvsktt-ner-sft',             'dvsktt-ner-sft'),
    ('thnhcngl/dvsktt-sft-best-checkpoint', 'dvsktt-sft-best-checkpoint'),
]

for ds, name in datasets:
    dest = f'/content/data/{name}'
    if os.path.exists(dest) and len(os.listdir(dest)) > 0:
        print(f'Already exists: {dest}')
        continue
    print(f'Downloading {ds}...')
    r = subprocess.run(
        ['kaggle', 'datasets', 'download', ds, '-p', '/content/data', '--unzip'],
        capture_output=True, text=True
    )
    print(r.stdout[:200])
    if r.returncode != 0:
        print(f'ERROR: {r.stderr[:200]}')

# Set paths
han_data_path = '/content/data/dvsktt-han-pretrain'
sft_data_path = '/content/data/dvsktt-ner-sft'
sft_ckpt_path = '/content/data/dvsktt-sft-best-checkpoint'

# Verify
for p in [han_data_path, sft_data_path, sft_ckpt_path]:
    if os.path.exists(p):
        print(f'OK: {p} → {os.listdir(p)}')
    else:
        print(f'MISSING: {p}')

=== /content/data ===
  dvsktt_han.txt
  dvsktt_han_merged.txt
  dvsktt-han-pretrain
  dvsktt-ner-sft
  dvsktt-sft-best-checkpoint
Dataset URL: https://www.kaggle.com/datasets/thnhcngl/dvsktt-h
Dataset URL: https://www.kaggle.com/datasets/thnhcngl/dvsktt-n
Dataset URL: https://www.kaggle.com/datasets/thnhcngl/dvsktt-s
OK: /content/data/dvsktt-han-pretrain → []
OK: /content/data/dvsktt-ner-sft → []
OK: /content/data/dvsktt-sft-best-checkpoint → []


## Load & Tokenize Corpus

In [27]:
random.seed(42)

corpus_path = '/content/data/dvsktt_han_merged.txt'
with open(corpus_path, 'r', encoding='utf-8') as f:
    all_lines = [l.strip() for l in f if len(l.strip()) >= 10]

DVSKTT_SIZE = 1311
dvsktt_lines  = all_lines[:DVSKTT_SIZE]
chinese_lines = all_lines[DVSKTT_SIZE:]
chinese_sampled = random.sample(chinese_lines, CONFIG['pretrain_sample'] - DVSKTT_SIZE)
lines = dvsktt_lines + chinese_sampled
random.shuffle(lines)

pretrain_dataset = Dataset.from_dict({'text': lines})
logger.log(f'Corpus: {len(pretrain_dataset):,} lines (DVSKTT: {DVSKTT_SIZE}, Han: {len(chinese_sampled)})')

def tokenize_fn(examples):
    result = tokenizer(
        examples['text'],
        truncation = True,
        max_length = CONFIG['max_seq_len'],
        padding    = 'max_length',
    )
    result['labels'] = result['input_ids'].copy()
    return result

tokenized = pretrain_dataset.map(tokenize_fn, batched=True, remove_columns=['text'], num_proc=2)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, pad_to_multiple_of=8)
logger.log(f'Tokenized: {len(tokenized):,} examples')

[   476.1s] Corpus: 5,000 lines (DVSKTT: 1311, Han: 3689)


Map (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

[   479.8s] Tokenized: 5,000 examples


## Training Loop

In [30]:
EPOCHS     = CONFIG['pretrain_epochs']
BATCH      = CONFIG['pretrain_batch']
GRAD_ACCUM = CONFIG['pretrain_grad_accum']
SAVE_EVERY = 500

dataloader  = DataLoader(tokenized, batch_size=BATCH, shuffle=True, collate_fn=data_collator)
total_steps = len(dataloader) * EPOCHS
optimizer   = optim.AdamW(model.parameters(), lr=CONFIG['pretrain_lr'])
scheduler   = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = int(total_steps * 0.05),
    num_training_steps = total_steps,
)

logger.log(f'Total steps: {total_steps:,} | Resume from: {RESUME_STEP}')
logger.log(f'Estimated time: ~{total_steps * 3.5 / 3600:.1f}h')

model.train()
global_step = 0

for epoch in range(EPOCHS):
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        global_step += 1

        # Skip đã chạy rồi khi resume
        if global_step <= RESUME_STEP:
            if global_step % 200 == 0:
                logger.log(f'Skipping step {global_step}/{RESUME_STEP}...')
            continue

        batch   = {k: v.to(model.device) for k, v in batch.items()}
        outputs = model(**batch)
        loss    = outputs.loss / GRAD_ACCUM
        loss.backward()
        total_loss += outputs.loss.item()

        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        # Log mỗi 100 steps
        if global_step % 100 == 0:
            avg = total_loss / (step + 1)
            elapsed = time.time() - logger.start
            steps_done = global_step - RESUME_STEP
            eta = elapsed / steps_done * (total_steps - global_step) if steps_done > 0 else 0
            logger.log(
                f'Epoch {epoch+1} | Step {global_step}/{total_steps} | '
                f'Loss: {avg:.4f} | ETA: {eta/3600:.2f}h'
            )

        # Auto-save checkpoint mỗi SAVE_EVERY steps
        if global_step % SAVE_EVERY == 0:
            ckpt_path = f"{CONFIG['ckpt_dir']}/pretrain_step{global_step}"
            model.save_pretrained(ckpt_path)
            tokenizer.save_pretrained(ckpt_path)
            logger.save_state({
                'step': global_step,
                'epoch': epoch,
                'loss': total_loss / (step + 1),
                'ckpt': ckpt_path,
            }, 'pretrain_state')
            logger.log(f'Checkpoint saved: {ckpt_path}')

    logger.log(f'Epoch {epoch+1} done | Avg Loss: {total_loss/len(dataloader):.4f}')

# Final save
final_path = f"{CONFIG['ckpt_dir']}/pretrain_final"
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
logger.save_state({'step': total_steps, 'status': 'completed'}, 'pretrain_state')
logger.log(f'Pretrain complete! Saved: {final_path}')

[   710.4s] Total steps: 7,500 | Resume from: 0
[   710.4s] Estimated time: ~7.3h
[   758.8s] Epoch 1 | Step 100/7500 | Loss: 3.4885 | ETA: 15.60h
[   806.7s] Epoch 1 | Step 200/7500 | Loss: 3.4665 | ETA: 8.18h
[   855.0s] Epoch 1 | Step 300/7500 | Loss: 3.4781 | ETA: 5.70h
[   903.2s] Epoch 1 | Step 400/7500 | Loss: 3.4844 | ETA: 4.45h
[   951.4s] Epoch 1 | Step 500/7500 | Loss: 3.5137 | ETA: 3.70h


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dvsktt_ner/checkpoints/pretrain_step500/tokenizer_config.json.


[   953.3s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/pretrain_state.json
[   953.3s] Checkpoint saved: /content/drive/MyDrive/dvsktt_ner/checkpoints/pretrain_step500
[  1002.3s] Epoch 1 | Step 600/7500 | Loss: 3.5127 | ETA: 3.20h
[  1051.1s] Epoch 1 | Step 700/7500 | Loss: 3.4970 | ETA: 2.84h
[  1099.9s] Epoch 1 | Step 800/7500 | Loss: 3.4933 | ETA: 2.56h
[  1148.1s] Epoch 1 | Step 900/7500 | Loss: 3.4877 | ETA: 2.34h
[  1196.8s] Epoch 1 | Step 1000/7500 | Loss: 3.4838 | ETA: 2.16h


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dvsktt_ner/checkpoints/pretrain_step1000/tokenizer_config.json.


[  1198.1s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/pretrain_state.json
[  1198.1s] Checkpoint saved: /content/drive/MyDrive/dvsktt_ner/checkpoints/pretrain_step1000
[  1246.7s] Epoch 1 | Step 1100/7500 | Loss: 3.4758 | ETA: 2.01h
[  1295.2s] Epoch 1 | Step 1200/7500 | Loss: 3.4643 | ETA: 1.89h
[  1343.8s] Epoch 1 | Step 1300/7500 | Loss: 3.4463 | ETA: 1.78h
[  1391.9s] Epoch 1 | Step 1400/7500 | Loss: 3.4323 | ETA: 1.68h
[  1440.3s] Epoch 1 | Step 1500/7500 | Loss: 3.4157 | ETA: 1.60h


...

[  4114.1s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/pretrain_state.json
[  4114.1s] Checkpoint saved: /content/drive/MyDrive/dvsktt_ner/checkpoints/pretrain_step7000
[  4162.2s] Epoch 3 | Step 7100/7500 | Loss: 2.8111 | ETA: 0.07h
[  4210.4s] Epoch 3 | Step 7200/7500 | Loss: 2.8147 | ETA: 0.05h
[  4258.6s] Epoch 3 | Step 7300/7500 | Loss: 2.8179 | ETA: 0.03h
[  4306.8s] Epoch 3 | Step 7400/7500 | Loss: 2.8183 | ETA: 0.02h
[  4354.9s] Epoch 3 | Step 7500/7500 | Loss: 2.8152 | ETA: 0.00h


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dvsktt_ner/checkpoints/pretrain_step7500/tokenizer_config.json.


[  4356.2s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/pretrain_state.json
[  4356.2s] Checkpoint saved: /content/drive/MyDrive/dvsktt_ner/checkpoints/pretrain_step7500
[  4356.2s] Epoch 3 done | Avg Loss: 2.8152


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/dvsktt_ner/checkpoints/pretrain_final/tokenizer_config.json.


[  4357.5s] State saved: /content/drive/MyDrive/dvsktt_ner/logs/pretrain_state.json
[  4357.5s] Pretrain complete! Saved: /content/drive/MyDrive/dvsktt_ner/checkpoints/pretrain_final
